In [ ]:
path_to_dodem = '/Users/jmdunca2/do-dem/'
from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')

import HARP_and_age as haa
import nustar_utilities as nuutil
import lightcurves as lc
import visualize_dem_results as viz
import all_nu_analysis as ana

import pickle
import importlib

import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
import matplotlib.dates as mdates

from astropy.visualization import quantity_support
quantity_support()

with open('/Users/jmdunca2/do-dem/reference_files/all_targets_postghost_postshut.pickle', 'rb') as f:
    all_targets = pickle.load(f)


def mod_timestring(timestring):
    modts=(':').join(timestring.split('-'))
    modts=('-').join(modts.split('_'))
    return modts

In [ ]:
with open('/Users/jmdunca2/do-dem/reference_files/samesames.pickle', 'rb') as f:
    data = pickle.load(f)

samesames = data['same region lists']
grf = data['ghost ray flags']

filelist = ana.get_same_region_file_lists(samesames, all_targets)

filelist_x = ana.get_same_region_file_lists(samesames, all_targets, noxrt=True)


In [ ]:

xrtkeys = ['29-may-18_1', '12-apr-19', '13-apr-19', '29-jan-20', 
           '29-apr-21', '14-jan-21', '3-may-21_2', '30-jul-21_1', '30-jul-21_2', '17-nov-21_1']


In [ ]:
import os
import glob

maxs=[]
maxts=[]
filepairs = []
for f in filelist:
    f.sort()
    fnum=0
    for f_ in f:
        with open(f_, 'rb') as f:
            data = pickle.load(f)
        if len(data['dn_in']) > 9:
            fnum+=1
            #print(f_)
            stem = f_.split('/')[-1].split('_')[0:-4]
            sj = ('_').join(stem)
            #print(sj)
            res = glob.glob(os.path.dirname(f_)+'/'+sj+'*no_xrt*.pickle')
            #print(res)
            res.sort()
            if res:
                filepairs.append([res[1], f_])
                with open(res[1], 'rb') as f:
                    data2 = pickle.load(f)
                #print(data2.keys())
                maxs.append(np.array([data2['max'], data['max']]))
                maxts.append(np.array([data2['max_temp'], data['max_temp']]))

    if fnum > 0:
        print(fnum, f_)
    
fig = plt.figure(figsize=(10,5))
plt.hist(np.array(maxs)[:,0]/np.array(maxs)[:,1])  


fig = plt.figure(figsize=(10,5))
#plt.hist(np.array(maxts)[:,0]/np.array(maxts)[:,1])  
plt.scatter(np.array(maxts)[:,0], np.array(maxts)[:,1])

In [ ]:
len(filepairs)

In [ ]:
def compare_files(file1, file2):
    
    with open(file1, 'rb') as f:
        data1 = pickle.load(f)
    with open(file2, 'rb') as f:
        data2 = pickle.load(f)
        

    lower1 = np.array(data1['DEM'])-np.array(data1['edem'][0])
    upper1 = np.array(data1['DEM'])+np.array(data1['edem'][1])

    lower2 = np.array(data2['DEM'])-np.array(data2['edem'][0])
    upper2 = np.array(data2['DEM'])+np.array(data2['edem'][1])

    if np.any(lower1 > upper2):
        print(file1, file2)

    if np.any(lower2 > upper1):
        print(file1, file2)

for ff in filepairs:
    compare_files(ff[0], ff[1])

In [ ]:

#Make comparison plots for all the times:

importlib.reload(viz)


labelfontsize=20

for pp in filepairs:
    data1, timestring1, time1 = viz.load_DEM(pp[1])
    data2, timestring2, time2 = viz.load_DEM(pp[0])
    namest=pp[1].split('/')[6][11:]
    namest2=pp[0].split('/')[6][11:]

    fig = plt.figure(constrained_layout=True, figsize=(25,10))
    gs = fig.add_gridspec(5, 2)
    ax1 = fig.add_subplot(gs[0:5, 0])
    ax2 = fig.add_subplot(gs[1:4, 1])

    
    working_dir = '/Users/jmdunca2/do-dem/consistency_plots/'
    
    
    #data1['fill_color'] = 'green'
    #data2['fill_color'] = 'orange'
    importlib.reload(viz)


    viz.compare_DEMs(data1, data2, timestring1, timestring2, title1=namest+', '+mod_timestring(timestring1), 
                     title2=namest2+', '+mod_timestring(timestring2), 
                     plotMK=False, plot=True, working_dir=working_dir, input_ax=ax1, input_ax2=ax2)
    
    ax1.tick_params(axis='both', labelsize=17)
    #ax1.set_title('Comparison - Two Quiescent Times on '+namest, fontsize=labelfontsize)
    ax1.set_title('', fontsize=labelfontsize)

    #ax2.set_title('Residuals', fontsize=labelfontsize)
    ax2.set_title('', fontsize=labelfontsize)
    ax1.set_ylim([1e17, 2e29])
    ax1.legend(ncols=1, fontsize=labelfontsize/1.5)


    #plt.savefig('very_incons_ex.png', dpi=300)